# 48 — ChromaDB
**Goal:** Use ChromaDB for persistent vector storage with metadata filtering.

## 1. Setting Up ChromaDB

In [ ]:
import chromadb
from chromadb.config import Settings

# In-memory for development
client = chromadb.Client(Settings(anonymized_telemetry=False))
try:
    collection = client.create_collection("resumes")
    print(f"Collection 'resumes' created")
except:
    collection = client.get_collection("resumes")
    print("Collection already exists")

## 2. Adding Resumes with Metadata

In [ ]:
# Add resume embeddings with metadata
collection.add(
    documents=[
        "Senior data scientist with Python, NLP, TensorFlow. 5 years experience.",
        "Java backend engineer with Spring Boot, microservices, 3 years.",
        "Frontend developer with React, TypeScript, 2 years experience.",
    ],
    metadatas=[
        {"role": "data_scientist", "years": 5, "skills": "python,nlp,tensorflow"},
        {"role": "backend", "years": 3, "skills": "java,spring,microservices"},
        {"role": "frontend", "years": 2, "skills": "react,typescript"},
    ],
    ids=["resume_001", "resume_002", "resume_003"],
)
print(f"Collection has {collection.count()} documents")

## 3. Querying with Metadata Filtering

In [ ]:
# Query: find data scientists with similarity
results = collection.query(
    query_texts=["looking for NLP expert with Python"],
    n_results=2,
    where={"role": "data_scientist"},
)
print("Filtered query results:")
for i, (doc, metadata, distance) in enumerate(zip(
    results['documents'][0], results['metadatas'][0], results['distances'][0]
)):
    print(f"  {i+1}. [{metadata['role']}] {doc[:50]:50s} (dist: {distance:.3f})")

## 4. Updating and Deleting

In [ ]:
# Update a resume
collection.update(
    documents=["Senior data scientist with 6 years experience, Python, NLP."],
    ids=["resume_001"],
    metadatas=[{"role": "data_scientist", "years": 6, "skills": "python,nlp"}],
)
print(f"Updated resume_001. Collection count: {collection.count()}")

# Get by ID
result = collection.get(ids=["resume_001"], include=["documents", "metadatas"])
print(f"Get resume_001: {result['documents'][0][:40]}... [{result['metadatas'][0]}]")

## Summary: ChromaDB adds metadata filtering on top of vector search. Good for production MVPs.